# NEON SIMD

NEON là Advanced SIMD extension của kiến trúc do ARM Ltd. định nghĩa. Nó tồn tại trong mọi ARMv8-A CPU hiện đại (mobile, embedded Linux, camera SoC).

## NEON ở mức kiến trúc

### 1.1 Vector width

- 128-bit fixed width
- Không scalable (khác SVE)

Lane Layout:

![](image1.png)

$\rightarrow$ Điều này quyết định cách bạn Tile GEMM

### 1.2 Register file

Trong AArch64:
- 32 vector registers: V0-V31
- Mỗi register 128-bit
- Có thể truy cập dưới dạng:
    - Q (128-bit)
    - D (64-bit)
    - S (32-bit)
    - H (16-bit)
    - B (8-bit)

Ví dụ:

In [ ]:
Q0  = 128-bit
D0  = lower 64-bit của Q0

Edge AI implication:

- Register blocking phải khớp số V-register khả dụng.

### 1.3 Load/Store model (RISC)

NEON không tự truy cập memory trong arithmetic ops.

Bạn phải:
$$
Load → Compute → Store
$$

Điều này khiến:
- Memory bandwidth trở thành bottleneck chính
- Cache-aware tiling cực kỳ quan trọng

## NEON ở mức ML Workload

ML inference chủ yếu là:
- GEMM
- Convolution
- Depthwise conv
- Elementwise ops
- Quantized dot product

NEON ánh xạ trực tiếp vào các primitive này

## GEMM trên NEON (Core)

GEMM:

$$
C[M,N] = A[M,K] \times B[K,N]
$$

Trong INT8 inference:
- A, B : int8
- Accumulator: int32

Vì NEON có 16 lane int8:

$\rightarrow$ Một instruction có thể multiply 16 int8

### Dot Product (ARMv8.4+)

Instruction:

- $vdotq\_s32$

Thực hiện:
- 4 dot-product song song
-Mỗi dot-product = 4 $\times$ int8 accumulate

Edge AI insight:
- Dot-product instruction giảm số instruction $\approx$ 4x so với scalar MAC

### Register Blocking Strategy

Ví dụ tile $4\times4$:
- 4 accumulators (int32 vectors)
- Loop theo K với stride 16

Pseudo-kernel:

In [ ]:
Load 16 int8 from A
Load 16 int8 from B
vdot accumulate

Critical factors:
- Unroll K-Loop
- Giữ accumulators trong registers
- Không spill ra stack

## NHWC vs NCHW Trên NEON

NEON thích:
- Contiguous memory
- Vector-aligned data

Trong mobile inference $\rightarrow$ NHWC thường hiệu quả hơn vì:
- Channel là dimension cuối
- Có thể load 16 channel liên tiếp

Ví dụ:

In [ ]:
[N, H, W, C]
        ↑ contiguous

$\rightarrow$ load 16 channel bằng 1 $vld1q\_s8$

Trong NCHW thì Channel bị stride lớn:  $\rightarrow$ cache miss nhiều hơn

Edge AI conclusion:
- Layout quyết định SIMD efficiency

## Memory & Cache Awareness

NEON không mạnh nếu:
- Memory không align 16 byte
- Access pattern random
- Không tile theo L1 cache

Typical L1:
- 32-64KB

GEMM tile phải đảm bảo:
- Block A và B nằm trong L1
- Reuse data tối đa trước khi evict

Công thức thực tế:
$$
M\_block \times K\_block + K\_block \times N\_block < L1 size
$$

## Quantization & NEON

NEON hỗ trợ:
- int8 arithmetic
- widening multiply
- saturating add
- rounding shift

Quantized pipeline:

$$int8 → int32 \_ accumulate → requantize → int8$$

Requantization gồm:
- Multiply by scale (fixed-point)
- Right shift
- Clamp $[-128,127]$

NEON có instruction hỗ trợ:
- vqrdmulhq
- vqrshl
- vmax / vmin

Edge insight:
- Requantization tốn nhiều instruction nếu không tối ưu.

## FMA (Floating Point)

Cho FP32:

$$vmlaq\_f32$$

Thực hiện:

$$
acc = acc + (a \times b)
$$

1 instruction = 4 FMA

FMA throughput quyết định peak FLOPS

## Throughput vs Latency

Edge AI quan tâm:
- Low latency (camera pipeline)
- Real-time constraint

CPU + NEON tốt cho:
- Small model
- Batch = 1
- Low startup overhead

GPU thường có dispatch overhead cao hơn

## NEON vs NPU

![](image2.png)

NEON là baseline.

NPU là accelerator.

## What Edge AI Engineer Phải Hiểu

Bạn cần phải nắm:
1. Vector lane math

2. Register blocking

3. Cache tilling

4. Load/store alignment

5. Dot-product instruction

6. Quantized arithmetic pipeline

7. Loop unrolling

8. Memory bandwidth ceiling

Nếu không, bạn chỉ đang “dùng framework”, không phải tối ưu kernel.

## Mental Model Chuẩn

Hãy nghĩ:
- NEON = 16-lane mini vector processor trong mỗi ARM Core

$$
Edge\_AI\_Performance = Compute\_efficiency \times Memory\_locality \times Instruction\_throughput
$$